# Kaggle ResNet18 SimCLR: resnet18_covidqu

This notebook runs only `resnet18_covidqu` (SimCLR on real unlabeled COVID-QU). Pretraining and fine-tuning are separate so multiple Kaggle sessions can run/resume independently.


<a href="https://colab.research.google.com/github/tlinhevg05/contrastive-synthesis-medcls_CVProject/blob/main/notebooks/01_train_gans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [1]:
from pathlib import Path
import os
import shutil
import pandas as pd

print('Kaggle input exists:', Path('/kaggle/input').exists())
print('Kaggle working exists:', Path('/kaggle/working').exists())
!nvidia-smi

Kaggle input exists: True
Kaggle working exists: True
Sun Jun  7 04:01:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  

## 2. Clone or Pull Repository

In [2]:
REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/kaggle/working/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    %cd {REPO_ROOT}
    !git pull
else:
    %cd /kaggle/working
    !git clone {REPO_URL}
    %cd {REPO_ROOT}

print('REPO_ROOT:', REPO_ROOT)
!git rev-parse --short HEAD

/kaggle/working
Cloning into 'contrastive-synthesis-medcls_CVProject'...
remote: Enumerating objects: 21575, done.
remote: Counting objects: 100% (302/302), done.
remote: Compressing objects: 100% (180/180), done.
remote: Total 21575 (delta 197), reused 200 (delta 119), pack-reused 21273 (from 3)
Receiving objects: 100% (21575/21575), 622.59 MiB | 53.41 MiB/s, done.
Resolving deltas: 100% (226/226), done.
Updating files: 100% (21254/21254), done.
/kaggle/working/contrastive-synthesis-medcls_CVProject
REPO_ROOT: /kaggle/working/contrastive-synthesis-medcls_CVProject
40d0a68c


## 3. Install Minimal Dependencies

Kaggle already includes PyTorch. Install only lightweight packages used by the scripts.

In [3]:
!pip install -q timm scikit-learn matplotlib pandas Pillow

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

## 4. Edit Kaggle Dataset Paths

After adding your Kaggle Dataset to this notebook, edit these paths to match the folder names under `/kaggle/input`. The helper cell below links them into the repository as `data/processed/...`, so the existing scripts do not need path changes.

In [ ]:
from pathlib import Path
import os

DATA_ROOT = Path("/kaggle/input/datasets/tlinhevg05/medcls-cvproject/data")

LABELLED_SOURCE = DATA_ROOT / "processed/labelled_4232"
UNLABELLED_SOURCE = DATA_ROOT / "processed/unlabelled_16934"
SYNTHETIC_SOURCE = DATA_ROOT / "processed/synthetic_dcgan-20260530T212146Z-3-001/synthetic_dcgan"
MANIFEST_SOURCE = DATA_ROOT / "manifests"

OUTPUT_ROOT = Path("/kaggle/working/results/experiments")
PREVIOUS_EXPERIMENT_SOURCE = None

# 'resnet18_covidqu', 'resnet18_imagenet_covidqu', 'resnet18_covidqu_syn', 'resnet18_imagenet_covidqu_syn'
EXPERIMENT_ID = "resnet18_covidqu"

PRETRAIN_EPOCHS = 70  # total target epoch; resume from 5 -> run 6..10
FINETUNE_EPOCHS = None

print("LABELLED_SOURCE:", LABELLED_SOURCE, LABELLED_SOURCE.exists())
print("UNLABELLED_SOURCE:", UNLABELLED_SOURCE, UNLABELLED_SOURCE.exists())
print("SYNTHETIC_SOURCE:", SYNTHETIC_SOURCE, SYNTHETIC_SOURCE.exists())
print("MANIFEST_SOURCE:", MANIFEST_SOURCE, MANIFEST_SOURCE.exists())
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("EXPERIMENT_ID:", EXPERIMENT_ID)

LABELLED_SOURCE: /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/labelled_4232 True
UNLABELLED_SOURCE: /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/unlabelled_16934 True
SYNTHETIC_SOURCE: /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/synthetic_dcgan-20260530T212146Z-3-001/synthetic_dcgan True
MANIFEST_SOURCE: /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/manifests True
OUTPUT_ROOT: /kaggle/working/results/experiments
EXPERIMENT_ID: resnet18_covidqu


## 5. Link Data and Prepare Manifests

In [5]:
def replace_path(target: Path, source: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() or target.is_symlink():
        if target.is_symlink() or target.is_file():
            target.unlink()
        else:
            shutil.rmtree(target)
    if source.exists():
        os.symlink(source, target, target_is_directory=source.is_dir())
        print('Linked', target, '->', source)
    else:
        print('WARNING: source missing:', source)

%cd {REPO_ROOT}
replace_path(REPO_ROOT / 'data/processed/labelled_4232', LABELLED_SOURCE)
replace_path(REPO_ROOT / 'data/processed/unlabelled_16934', UNLABELLED_SOURCE)
replace_path(REPO_ROOT / 'data/processed/synthetic_dcgan', SYNTHETIC_SOURCE)

manifest_dir = REPO_ROOT / 'data/manifests'
manifest_dir.mkdir(parents=True, exist_ok=True)

for name in ['train.csv', 'val.csv', 'test.csv', 'labelled_all.csv', 'split_summary.json']:
    src = MANIFEST_SOURCE / name
    dst = manifest_dir / name
    if src.exists():
        shutil.copy2(src, dst)
        print('Copied manifest:', dst)
    elif dst.exists():
        print('Using repo manifest:', dst)
    else:
        print('WARNING: missing manifest:', src)

# Normalize synthetic manifest for Kaggle. Stage 1 manifests made on Colab may contain Drive absolute paths.
src_syn_manifest = MANIFEST_SOURCE / 'synthetic_dcgan.csv'
dst_syn_manifest = manifest_dir / 'synthetic_dcgan.csv'
if src_syn_manifest.exists():
    df = pd.read_csv(src_syn_manifest)
    normalized_paths = []
    for _, row in df.iterrows():
        original = Path(str(row['image_path']))
        class_name = row['class_name']
        filename = original.name
        candidates = [
            Path('data/processed/synthetic_dcgan') / class_name / 'images' / filename,
            Path('data/processed/synthetic_dcgan') / class_name / filename,
            Path('data/processed/synthetic_dcgan') / original.name,
        ]
        selected = candidates[0]
        for candidate in candidates:
            if (REPO_ROOT / candidate).exists():
                selected = candidate
                break
        normalized_paths.append(str(selected))
    df['image_path'] = normalized_paths
    df.to_csv(dst_syn_manifest, index=False)
    print('Wrote Kaggle-normalized synthetic manifest:', dst_syn_manifest)
elif dst_syn_manifest.exists():
    print('Using repo synthetic manifest:', dst_syn_manifest)
else:
    rows = []
    class_to_label = {'COVID': 0, 'Lung_Opacity': 1, 'Viral_Pneumonia': 2, 'Normal': 3}
    image_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    if SYNTHETIC_SOURCE.exists():
        for class_name, label in class_to_label.items():
            class_dir = REPO_ROOT / 'data/processed/synthetic_dcgan' / class_name
            search_root = class_dir / 'images' if (class_dir / 'images').exists() else class_dir
            for image_path in sorted(search_root.rglob('*')):
                if image_path.is_file() and image_path.suffix.lower() in image_exts:
                    rows.append({
                        'image_path': str(image_path.relative_to(REPO_ROOT)),
                        'class_name': class_name,
                        'label': label,
                        'source': 'synthetic',
                        'generator': 'dcgan',
                    })
    if rows:
        pd.DataFrame(rows).to_csv(dst_syn_manifest, index=False)
        print('Generated synthetic manifest from Kaggle synthetic folder:', dst_syn_manifest, 'rows=', len(rows))
    else:
        print('WARNING: synthetic_dcgan.csv not found and synthetic images could not be discovered. Synthetic experiments will fail until this is provided.')

!find data -maxdepth 3 -type d | sort | head -40
!ls -lh data/manifests

/kaggle/working/contrastive-synthesis-medcls_CVProject
Linked /kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/labelled_4232 -> /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/labelled_4232
Linked /kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/unlabelled_16934 -> /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/unlabelled_16934
Linked /kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/synthetic_dcgan -> /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/synthetic_dcgan-20260530T212146Z-3-001/synthetic_dcgan
Copied manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests/train.csv
Copied manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests/val.csv
Copied manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests/test.csv
Copied manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests

## 6. Verify Inputs and Scripts

In [6]:
SYNTHETIC_MANIFEST = REPO_ROOT / 'data/manifests/synthetic_dcgan.csv'

!python scripts/check_experiment_inputs.py --synthetic-manifest "{SYNTHETIC_MANIFEST}"
!python -m py_compile scripts/run_simclr_resnet.py scripts/run_classification_resnet.py
!python scripts/run_simclr_resnet.py --help | grep resume || true


Experiment Input Check Report
[PASS] common config
  - loaded configs/experiments/common.yaml
[PASS] fixed supervised manifests
  - train: {'COVID': 578, 'Lung_Opacity': 961, 'Viral_Pneumonia': 215, 'Normal': 1630} total=3384
  - val: {'COVID': 72, 'Lung_Opacity': 120, 'Viral_Pneumonia': 26, 'Normal': 203} total=421
  - test: {'COVID': 73, 'Lung_Opacity': 121, 'Viral_Pneumonia': 28, 'Normal': 205} total=427
[PASS] experiment config files
[PASS] resnet18_covidqu
  - planned output_dir: results/experiments/resnet18_covidqu
[PASS] resnet18_covidqu_syn
  - synthetic_dcgan: {'COVID': 1000, 'Lung_Opacity': 1000, 'Viral_Pneumonia': 1000, 'Normal': 1000} total=4000
  - planned output_dir: results/experiments/resnet18_covidqu_syn
[PASS] resnet18_imagenet
  - no contrastive pretraining data required
  - planned output_dir: results/experiments/resnet18_imagenet
[PASS] resnet18_imagenet_covidqu
  - planned output_dir: results/experiments/resnet18_imagenet_covidqu
[PASS] resnet18_imagenet_covidqu_

## 7. Experiment Settings: resnet18_covidqu

`PRETRAIN_EPOCHS` is the total target epoch. Increase it from 10 to 20, 30, and so on to resume in chunks.


In [7]:
EXP = 'resnet18_covidqu'
CONFIG = 'configs/experiments/resnet18/covidqu.yaml'
USES_SYNTHETIC = False
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_simclr_backbone.pth'
RESUME_CKPT = OUT / 'pretrain/checkpoints/last_simclr_checkpoint.pth'

pretrain_epoch_arg = f'--epochs {PRETRAIN_EPOCHS}' if PRETRAIN_EPOCHS is not None else ''
finetune_epoch_arg = f'--epochs {FINETUNE_EPOCHS}' if FINETUNE_EPOCHS is not None else ''

print('EXP:', EXP)
print('CONFIG:', CONFIG)
print('OUT:', OUT)
print('CKPT:', CKPT, 'exists=', CKPT.exists())
print('RESUME_CKPT:', RESUME_CKPT, 'exists=', RESUME_CKPT.exists())
print('USES_SYNTHETIC:', USES_SYNTHETIC)
print('pretrain_epoch_arg:', pretrain_epoch_arg)
print('finetune_epoch_arg:', finetune_epoch_arg)

EXP: resnet18_covidqu
CONFIG: configs/experiments/resnet18/covidqu.yaml
OUT: /kaggle/working/results/experiments/resnet18_covidqu
CKPT: /kaggle/working/results/experiments/resnet18_covidqu/pretrain/checkpoints/best_simclr_backbone.pth exists= False
RESUME_CKPT: /kaggle/working/results/experiments/resnet18_covidqu/pretrain/checkpoints/last_simclr_checkpoint.pth exists= False
USES_SYNTHETIC: False
pretrain_epoch_arg: --epochs 70
finetune_epoch_arg: 


<a href="https://colab.research.google.com/github/tlinhevg05/contrastive-synthesis-medcls_CVProject/blob/main/notebooks/01_train_gans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


<a href="https://colab.research.google.com/github/tlinhevg05/contrastive-synthesis-medcls_CVProject/blob/main/notebooks/01_train_gans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## 8. Restore Previous Kaggle Result

If you uploaded a previous output folder as a Kaggle Dataset, restore it before pretraining so SimCLR can resume.


In [8]:
from pathlib import Path
import zipfile

zip_candidates = list(Path("/kaggle/input").rglob("resnet18_covidqu_results.zip"))
print("zip candidates:", zip_candidates)

if not zip_candidates:
    raise FileNotFoundError(
        "Could not find resnet18_covidqu_results.zip in /kaggle/input. "
        "Add the previous Kaggle output zip first."
    )

ZIP_PATH = zip_candidates[0]
print("Using ZIP:", ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall("/kaggle/working")

RESUME_CKPT = OUT / "pretrain/checkpoints/last_simclr_checkpoint.pth"
CKPT = OUT / "pretrain/checkpoints/best_simclr_backbone.pth"

print("OUT exists:", OUT.exists(), OUT)
print("RESUME_CKPT exists:", RESUME_CKPT.exists(), RESUME_CKPT)
print("CKPT exists:", CKPT.exists(), CKPT)

if not RESUME_CKPT.exists():
    raise FileNotFoundError(f"Resume checkpoint not found: {RESUME_CKPT}")

zip candidates: [PosixPath('/kaggle/input/notebooks/maidngtrn/notebookb95b69b45c/resnet18_covidqu_results.zip')]
Using ZIP: /kaggle/input/notebooks/maidngtrn/notebookb95b69b45c/resnet18_covidqu_results.zip
OUT exists: True /kaggle/working/results/experiments/resnet18_covidqu
RESUME_CKPT exists: True /kaggle/working/results/experiments/resnet18_covidqu/pretrain/checkpoints/last_simclr_checkpoint.pth
CKPT exists: True /kaggle/working/results/experiments/resnet18_covidqu/pretrain/checkpoints/best_simclr_backbone.pth


## 9. Pretrain Only

Run this cell repeatedly by increasing `PRETRAIN_EPOCHS`. If it prints `Epoch 1/...` when you expected resume, stop and check `RESUME_CKPT`.


In [9]:
print('RESUME_CKPT:', RESUME_CKPT, 'exists=', RESUME_CKPT.exists())
!python scripts/run_simclr_resnet.py \
  --config "{CONFIG}" \
  --real-unlabeled-dir data/processed/unlabelled_16934 \
  --output-dir "{OUT}" \
  --resume-checkpoint "{RESUME_CKPT}" \
  {pretrain_epoch_arg}


RESUME_CKPT: /kaggle/working/results/experiments/resnet18_covidqu/pretrain/checkpoints/last_simclr_checkpoint.pth exists= True
Resuming SimCLR from /kaggle/working/results/experiments/resnet18_covidqu/pretrain/checkpoints/last_simclr_checkpoint.pth at epoch 55
Epoch 56/70 simclr_loss=3.0580
Epoch 57/70 simclr_loss=3.0584
Epoch 58/70 simclr_loss=3.0657
Epoch 59/70 simclr_loss=3.0625
Epoch 60/70 simclr_loss=3.0577
Epoch 61/70 simclr_loss=3.0593
Epoch 62/70 simclr_loss=3.0582
Epoch 63/70 simclr_loss=3.0580
Epoch 64/70 simclr_loss=3.0546
Epoch 65/70 simclr_loss=3.0526
Epoch 66/70 simclr_loss=3.0534
Epoch 67/70 simclr_loss=3.0518
Epoch 68/70 simclr_loss=3.0520
Epoch 69/70 simclr_loss=3.0508
Epoch 70/70 simclr_loss=3.0443
Saved SimCLR checkpoint: /kaggle/working/results/experiments/resnet18_covidqu/pretrain/checkpoints/best_simclr_backbone.pth


## 10. Fine-Tune Only

Run after pretraining reaches the epoch target you want to report.


In [10]:
print('CKPT:', CKPT, 'exists=', CKPT.exists())
if not CKPT.exists():
    raise FileNotFoundError(f'SimCLR checkpoint not found: {CKPT}. Finish pretraining first.')
!python scripts/run_classification_resnet.py \
  --config "{CONFIG}" \
  --manifest-dir data/manifests \
  --output-dir "{OUT}" \
  --pretrained-checkpoint "{CKPT}" \
  {finetune_epoch_arg}


CKPT: /kaggle/working/results/experiments/resnet18_covidqu/pretrain/checkpoints/best_simclr_backbone.pth exists= True
Missing keys after SimCLR encoder load: ['fc.weight', 'fc.bias']
Epoch 1/70 train_loss=1.1996 val_loss=1.0836 val_acc=0.5772 val_f1_macro=0.4006
Epoch 2/70 train_loss=1.0112 val_loss=0.9266 val_acc=0.6888 val_f1_macro=0.5366
Epoch 3/70 train_loss=0.8749 val_loss=0.8189 val_acc=0.7411 val_f1_macro=0.6593
Epoch 4/70 train_loss=0.7793 val_loss=0.7373 val_acc=0.7577 val_f1_macro=0.6997
Epoch 5/70 train_loss=0.7007 val_loss=0.6700 val_acc=0.7720 val_f1_macro=0.7247
Epoch 6/70 train_loss=0.6390 val_loss=0.6183 val_acc=0.7862 val_f1_macro=0.7475
Epoch 7/70 train_loss=0.5885 val_loss=0.5754 val_acc=0.8005 val_f1_macro=0.7694
Epoch 8/70 train_loss=0.5555 val_loss=0.5448 val_acc=0.8147 val_f1_macro=0.7923
Epoch 9/70 train_loss=0.5131 val_loss=0.5191 val_acc=0.8266 val_f1_macro=0.8069
Epoch 10/70 train_loss=0.4768 val_loss=0.4893 val_acc=0.8337 val_f1_macro=0.8141
Epoch 11/70 trai

## 11. Display and Package Results


In [11]:
import json

metrics_path = OUT / 'metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    display(pd.DataFrame([{**{'experiment_id': EXP}, **metrics}]))
else:
    print('WARNING: metrics.json not found:', metrics_path)

!find "{OUT}" -maxdepth 4 -type f | sort
!cd /kaggle/working && zip -qr "{EXP}_results.zip" results/experiments/"{EXP}"
print('Result zip:', Path('/kaggle/working') / f'{EXP}_results.zip')


,experiment_id,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,best_epoch,best_val_f1_macro
0,resnet18_covidqu,0.911007,0.929762,0.922137,0.925799,0.910884,0.911007,0.910795,57,0.88953


/kaggle/working/results/experiments/resnet18_covidqu/best_checkpoint.pth
/kaggle/working/results/experiments/resnet18_covidqu/classification_report.csv
/kaggle/working/results/experiments/resnet18_covidqu/config_resolved_simclr.yaml
/kaggle/working/results/experiments/resnet18_covidqu/config_resolved.yaml
/kaggle/working/results/experiments/resnet18_covidqu/confusion_matrix.png
/kaggle/working/results/experiments/resnet18_covidqu/metrics.json
/kaggle/working/results/experiments/resnet18_covidqu/pretrain/checkpoints/best_simclr_backbone.pth
/kaggle/working/results/experiments/resnet18_covidqu/pretrain/checkpoints/last_simclr_checkpoint.pth
/kaggle/working/results/experiments/resnet18_covidqu/pretrain/simclr_history.json
Result zip: /kaggle/working/resnet18_covidqu_results.zip
